###### 0from kaggle_secrets import UserSecretsClient
import numpy as np
from PIL import Image
import json, matplotlib.pyplot as plt
import tensorflow as tf

SAVE_DIR = '/kaggle/working/food_model_v2'

with open(f'{SAVE_DIR}/class_names_filtered.json') as f:
    class_names = json.load(f)

# Upload image
from kaggle_datasets import KaggleDatasets
import ipywidgets as widgets
from IPython.display import display

upload = widgets.FileUpload(accept='image/*', multiple=False)
display(upload)

In [4]:
import tensorflow as tf
from tensorflow.keras import mixed_precision
import os

mixed_precision.set_global_policy('mixed_float16')

SAVE_DIR = '/kaggle/working/food_model_v4'
os.makedirs(SAVE_DIR, exist_ok=True)

print(f"TF Version: {tf.__version__}")
print(f"GPU: {tf.config.list_physical_devices('GPU')}")
print(f"Policy: {mixed_precision.global_policy()}")
print(f"Save dir: {SAVE_DIR}")

2026-06-16 12:34:11.541182: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1781613251.722813      58 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1781613251.773434      58 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1781613252.229498      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781613252.229539      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781613252.229542      58 computation_placer.cc:177] computation placer alr

TF Version: 2.19.0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Policy: <DTypePolicy "mixed_float16">
Save dir: /kaggle/working/food_model_v4


In [5]:
import os
import json
import numpy as np
import tensorflow as tf

SAVE_DIR = '/kaggle/working/food_model_v4'

# ── Paths ──
FOOD101_DIR = '/kaggle/input/datasets/kmader/food41/images'
ARABIC_DIR  = '/kaggle/input/datasets/araraltawil/arabic-food-101/Arabic Food 101 V3/Images'
EGYPTIAN_DIR = '/kaggle/input/datasets/abdullahmohammed29/egyptian-food/egyptian food'
FRUIT_DIR   = '/kaggle/input/datasets/araraltawil/fruit-101-dataset'

# ── Step 1: Food-101 filtered classes ──
REMOVE_FOODS = [
    "edamame", "poutine", "peking_duck", "bibimbap", "gyoza",
    "takoyaki", "sashimi", "pierogi", "beet_salad", "ceviche",
    "pho", "ramen", "pad_thai", "clam_chowder", "pulled_pork_sandwich",
    "baby_back_ribs", "pork_chop", "hot_and_sour_soup",
    "dumplings", "french_toast", "eggs_benedict", "escargots"
]

with open('/kaggle/input/datasets/kmader/food41/meta/meta/classes.txt') as f:
    food101_classes = [l.strip() for l in f if l.strip() not in REMOVE_FOODS]

print(f"Food-101 classes: {len(food101_classes)}")

# ── Step 2: Arabic Food 101 classes ──
# Map Arabic Food 101 class names to standardized names
ARABIC_CLASS_MAP = {
    'Fava Beans': 'fava_beans',
    'muskhan': 'muskhan',
    'mojadra': 'mojadra',
    'plate of meat': 'plate_of_meat',
    'Qatayef': 'qatayef',
    'Falafel': 'falafel',
    'Makmoura': 'makmoura',
    'Shishabark': 'shishabark',
    'Mansaf': 'mansaf',
    'Kunafa': 'kunafa',
    'kebab': 'kebab',
    'Rice with milk': 'rice_with_milk',
    'Kibbeh': 'kibbeh',
    'Mandi': 'mandi',
    'Apple pie': 'apple_pie',
    'pastries': 'pastries',
    'Mahashi': 'mahshi',
    'Shawarma': 'shawarma',
    'koshari': 'koshari',
    'samosas': 'samosa',
}

# ── Step 3: Egyptian food classes ──
EGYPTIAN_CLASS_MAP = {
    'Bamya': 'bamya',
    'Fattah': 'fattah',
    'Kunafa': 'kunafa',
    'Molokhiya': 'molokhia',
    'Koshary': 'koshari',
}

# ── Step 4: Fruit classes ──
FRUIT_CLASSES = [
    'orange', 'Apple', 'ressberry', 'mango fruit',
    'avocado', 'chery fruit', 'grapes', 'banana'
]
FRUIT_CLASS_MAP = {f: f.lower().replace(' ', '_') for f in FRUIT_CLASSES}

# ── Step 5: Build master class list ──
# Start with Food-101
all_classes = list(food101_classes)

# Add Arabic food classes that don't overlap with Food-101
arabic_new = []
for orig, std in ARABIC_CLASS_MAP.items():
    if std not in all_classes:
        arabic_new.append(std)
        all_classes.append(std)

# Add Egyptian food classes that don't overlap
egyptian_new = []
for orig, std in EGYPTIAN_CLASS_MAP.items():
    if std not in all_classes:
        egyptian_new.append(std)
        all_classes.append(std)

# Add fruit classes
fruit_new = []
for orig, std in FRUIT_CLASS_MAP.items():
    if std not in all_classes:
        fruit_new.append(std)
        all_classes.append(std)

print(f"Food-101 classes  : {len(food101_classes)}")
print(f"Arabic new classes: {len(arabic_new)} → {arabic_new}")
print(f"Egyptian new      : {len(egyptian_new)} → {egyptian_new}")
print(f"Fruit new         : {len(fruit_new)}")
print(f"Total classes     : {len(all_classes)}")

# Save class names
with open(f'{SAVE_DIR}/class_names.json', 'w') as f:
    json.dump(all_classes, f, indent=2)
print("✅ Saved class names!")

# ── Step 6: Build image paths ──
all_paths, all_labels = [], []

# Food-101
for cls in food101_classes:
    idx = all_classes.index(cls)
    cls_dir = f'{FOOD101_DIR}/{cls}'
    if os.path.exists(cls_dir):
        for f in os.listdir(cls_dir):
            if f.lower().endswith(('.jpg', '.jpeg', '.png')):
                all_paths.append(f'{cls_dir}/{f}')
                all_labels.append(idx)

food101_count = len(all_paths)
print(f"\nFood-101 images: {food101_count}")

# Arabic Food 101 — merge overlapping classes
for orig, std in ARABIC_CLASS_MAP.items():
    idx = all_classes.index(std)
    cls_dir = f'{ARABIC_DIR}/{orig}'
    if os.path.exists(cls_dir):
        for f in os.listdir(cls_dir):
            if f.lower().endswith(('.jpg', '.jpeg', '.png')):
                all_paths.append(f'{cls_dir}/{f}')
                all_labels.append(idx)

arabic_count = len(all_paths) - food101_count
print(f"Arabic Food images: {arabic_count}")

# Egyptian food — merge overlapping classes
after_arabic = len(all_paths)
for orig, std in EGYPTIAN_CLASS_MAP.items():
    idx = all_classes.index(std)
    cls_dir = f'{EGYPTIAN_DIR}/{orig}'
    if os.path.exists(cls_dir):
        for f in os.listdir(cls_dir):
            if f.lower().endswith(('.jpg', '.jpeg', '.png')):
                all_paths.append(f'{cls_dir}/{f}')
                all_labels.append(idx)

egyptian_count = len(all_paths) - after_arabic
print(f"Egyptian food images: {egyptian_count}")

# Fruit dataset
after_egyptian = len(all_paths)
for orig, std in FRUIT_CLASS_MAP.items():
    idx = all_classes.index(std)
    cls_dir = f'{FRUIT_DIR}/{orig}'
    if os.path.exists(cls_dir):
        for f in os.listdir(cls_dir):
            if f.lower().endswith(('.jpg', '.jpeg', '.png')):
                all_paths.append(f'{cls_dir}/{f}')
                all_labels.append(idx)

fruit_count = len(all_paths) - after_egyptian
print(f"Fruit images: {fruit_count}")

print(f"\n✅ Total images: {len(all_paths)}")
print(f"✅ Total classes: {len(all_classes)}")

# Train/val split
all_paths = np.array(all_paths)
all_labels_cat = tf.keras.utils.to_categorical(all_labels, len(all_classes))

np.random.seed(42)
idx = np.random.permutation(len(all_paths))
split = int(0.8 * len(idx))
train_idx, val_idx = idx[:split], idx[split:]

print(f"Train: {len(train_idx)} | Val: {len(val_idx)}")

Food-101 classes: 80
Food-101 classes  : 80
Arabic new classes: 17 → ['fava_beans', 'muskhan', 'mojadra', 'plate_of_meat', 'qatayef', 'makmoura', 'shishabark', 'mansaf', 'kunafa', 'kebab', 'rice_with_milk', 'kibbeh', 'mandi', 'pastries', 'mahshi', 'shawarma', 'koshari']
Egyptian new      : 3 → ['bamya', 'fattah', 'molokhia']
Fruit new         : 8
Total classes     : 108
✅ Saved class names!

Food-101 images: 80000
Arabic Food images: 3333
Egyptian food images: 504
Fruit images: 1473

✅ Total images: 85310
✅ Total classes: 108
Train: 68248 | Val: 17062


In [12]:
IMG_SIZE = 224
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE
NUM_CLASSES = len(all_classes)

def load_image(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    img = tf.cast(img, tf.float32)
    return img, label

def augment(img, label):
    # Random crop from slightly larger image
    img = tf.image.resize(img, [IMG_SIZE + 32, IMG_SIZE + 32])
    img = tf.image.random_crop(img, [IMG_SIZE, IMG_SIZE, 3])
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_brightness(img, 0.3)
    img = tf.image.random_contrast(img, 0.7, 1.3)
    img = tf.image.random_saturation(img, 0.7, 1.3)
    img = tf.image.random_hue(img, 0.1)
    img = tf.clip_by_value(img, 0, 255)
    return img, label

def mixup(image, label, alpha=0.2):
    batch_size = tf.shape(image)[0]
    indices = tf.random.shuffle(tf.range(batch_size))
    lam = tf.cast(tf.random.uniform([], 0, alpha), tf.float32)
    label = tf.cast(label, tf.float32)
    mixed_image = lam * image + (1 - lam) * tf.gather(image, indices)
    mixed_label = lam * label + (1 - lam) * tf.gather(label, indices)
    return mixed_image, mixed_label

train_ds = tf.data.Dataset.from_tensor_slices(
    (all_paths[train_idx], all_labels_cat[train_idx])
).map(load_image, num_parallel_calls=AUTOTUNE
).map(augment, num_parallel_calls=AUTOTUNE
).shuffle(3000
).batch(BATCH_SIZE
).map(mixup, num_parallel_calls=AUTOTUNE
).prefetch(AUTOTUNE)

val_ds = tf.data.Dataset.from_tensor_slices(
    (all_paths[val_idx], all_labels_cat[val_idx])
).map(load_image, num_parallel_calls=AUTOTUNE
).batch(BATCH_SIZE
).prefetch(AUTOTUNE)

print(f"✅ Pipeline ready!")
print(f"Classes: {NUM_CLASSES}")
print(f"Train batches: {len(train_ds)}")
print(f"Val batches  : {len(val_ds)}")

✅ Pipeline ready!
Classes: 108
Train batches: 2133
Val batches  : 534


In [13]:
from tensorflow.keras import layers, models
from tensorflow.keras.applications import EfficientNetV2M

print("Building EfficientNetV2M...")

base_model = EfficientNetV2M(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights="imagenet",
    include_preprocessing=True,
)
base_model.trainable = False

inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(1024, activation="relu")(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.5)(x)
x = layers.Dense(512, activation="relu")(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.4)(x)
x = layers.Dense(256, activation="relu")(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(NUM_CLASSES, activation="softmax", dtype='float32')(x)

model = models.Model(inputs, outputs)
print(f"✅ Model ready!")
print(f"Params: {model.count_params():,}")
print(f"Classes: {NUM_CLASSES}")

Building EfficientNetV2M...
✅ Model ready!
Params: 55,152,160
Classes: 108


In [ ]:
from tensorflow.keras import optimizers, callbacks
import os

SAVE_DIR = '/kaggle/working/food_model_v4'
steps_per_epoch = len(train_idx) // BATCH_SIZE

# ════════════════════════════════
# PHASE 1: Train head — 15 epochs
# ════════════════════════════════
print("\n" + "="*60)
print("PHASE 1: Training head (base frozen) — 15 epochs")
print("="*60)

lr1 = tf.keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=1e-3,
    decay_steps=steps_per_epoch * 15,
    warmup_steps=steps_per_epoch * 3,
    alpha=1e-6
)

model.compile(
    optimizer=tf.keras.optimizers.AdamW(
        learning_rate=lr1,
        weight_decay=1e-4
    ),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=["accuracy",
             tf.keras.metrics.TopKCategoricalAccuracy(k=5, name="top5")]
)

history1 = model.fit(
    train_ds, epochs=15, validation_data=val_ds,
    callbacks=[
        callbacks.ModelCheckpoint(f"{SAVE_DIR}/phase1_best.keras",
            monitor="val_accuracy", save_best_only=True, verbose=1),
        callbacks.ModelCheckpoint(f"{SAVE_DIR}/phase1_latest.keras",
            save_best_only=False, verbose=0, save_freq='epoch'),
        callbacks.EarlyStopping(monitor="val_accuracy",
            patience=5, restore_best_weights=True, verbose=1),
        callbacks.CSVLogger(f"{SAVE_DIR}/phase1_log.csv"),
    ]
)

best_p1 = max(history1.history['val_accuracy'])
print(f"\n✅ Phase 1: {best_p1*100:.2f}%")

# ════════════════════════════════
# PHASE 2: Fine-tune top 200 layers — 25 epochs
# ════════════════════════════════
print("\n" + "="*60)
print("PHASE 2: Fine-tuning top 200 layers — 25 epochs")
print("="*60)

model = tf.keras.models.load_model(f'{SAVE_DIR}/phase1_best.keras')

efficientnet_layer = model.layers[1]
efficientnet_layer.trainable = True
total_layers = len(efficientnet_layer.layers)
for i, layer in enumerate(efficientnet_layer.layers):
    layer.trainable = i >= (total_layers - 200)

unfrozen = sum(1 for l in efficientnet_layer.layers if l.trainable)
print(f"Unfrozen: {unfrozen}/{total_layers}")

lr2 = tf.keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=3e-5,
    decay_steps=steps_per_epoch * 25,
    warmup_steps=steps_per_epoch * 3,
    alpha=1e-8
)

model.compile(
    optimizer=tf.keras.optimizers.AdamW(
        learning_rate=lr2,
        weight_decay=5e-5
    ),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.05),
    metrics=["accuracy",
             tf.keras.metrics.TopKCategoricalAccuracy(k=5, name="top5")]
)

history2 = model.fit(
    train_ds, epochs=25, validation_data=val_ds,
    callbacks=[
        callbacks.ModelCheckpoint(f"{SAVE_DIR}/phase2_best.keras",
            monitor="val_accuracy", save_best_only=True, verbose=1),
        callbacks.ModelCheckpoint(f"{SAVE_DIR}/phase2_latest.keras",
            save_best_only=False, verbose=0, save_freq='epoch'),
        callbacks.EarlyStopping(monitor="val_accuracy",
            patience=8, restore_best_weights=True, verbose=1),
        callbacks.CSVLogger(f"{SAVE_DIR}/phase2_log.csv"),
    ]
)

best_p2 = max(history2.history['val_accuracy'])
print(f"\n✅ Phase 2: {best_p2*100:.2f}%")

# ════════════════════════════════
# PHASE 3: Full fine-tune — 15 epochs
# ════════════════════════════════
print("\n" + "="*60)
print("PHASE 3: Full fine-tuning all layers — 15 epochs")
print("="*60)

model = tf.keras.models.load_model(f'{SAVE_DIR}/phase2_best.keras')

efficientnet_layer = model.layers[1]
efficientnet_layer.trainable = True
for layer in efficientnet_layer.layers:
    layer.trainable = True

print(f"All {len(efficientnet_layer.layers)} layers unfrozen!")

lr3 = tf.keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=5e-6,
    decay_steps=steps_per_epoch * 15,
    warmup_steps=steps_per_epoch * 2,
    alpha=1e-9
)

model.compile(
    optimizer=tf.keras.optimizers.AdamW(
        learning_rate=lr3,
        weight_decay=1e-5
    ),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.03),
    metrics=["accuracy",
             tf.keras.metrics.TopKCategoricalAccuracy(k=5, name="top5")]
)

history3 = model.fit(
    train_ds, epochs=15, validation_data=val_ds,
    callbacks=[
        callbacks.ModelCheckpoint(f"{SAVE_DIR}/final_best.keras",
            monitor="val_accuracy", save_best_only=True, verbose=1),
        callbacks.ModelCheckpoint(f"{SAVE_DIR}/final_latest.keras",
            save_best_only=False, verbose=0, save_freq='epoch'),
        callbacks.EarlyStopping(monitor="val_accuracy",
            patience=8, restore_best_weights=True, verbose=1),
        callbacks.CSVLogger(f"{SAVE_DIR}/phase3_log.csv"),
    ]
)

best_p3 = max(history3.history['val_accuracy'])
print(f"\n✅ Phase 3: {best_p3*100:.2f}%")
print(f"\n{'='*60}")
print(f"FINAL SUMMARY")
print(f"{'='*60}")
print(f"Phase 1 : {best_p1*100:.2f}%")
print(f"Phase 2 : {best_p2*100:.2f}%")
print(f"Phase 3 : {best_p3*100:.2f}%")
print(f"Top-5   : {max(history3.history['val_top5'])*100:.2f}%")


PHASE 1: Training head (base frozen) — 15 epochs
Epoch 1/15
2133/2133 ━━━━━━━━━━━━━━━━━━━━ 0s 121ms/step - accuracy: 0.2118 - loss: 3.9710 - top5: 0.4464
Epoch 1: val_accuracy improved from None to 0.48787, saving model to /kaggle/working/food_model_v4/phase1_best.keras

Epoch 1: finished saving model to /kaggle/working/food_model_v4/phase1_best.keras
2133/2133 ━━━━━━━━━━━━━━━━━━━━ 471s 175ms/step - accuracy: 0.2818 - loss: 3.6076 - top5: 0.5541 - val_accuracy: 0.4879 - val_loss: 2.4905 - val_top5: 0.7864
Epoch 2/15
2133/2133 ━━━━━━━━━━━━━━━━━━━━ 0s 95ms/step - accuracy: 0.3715 - loss: 3.2249 - top5: 0.6698
Epoch 2: val_accuracy improved from 0.48787 to 0.51993, saving model to /kaggle/working/food_model_v4/phase1_best.keras

Epoch 2: finished saving model to /kaggle/working/food_model_v4/phase1_best.keras
2133/2133 ━━━━━━━━━━━━━━━━━━━━ 259s 119ms/step - accuracy: 0.3808 - loss: 3.2049 - top5: 0.6773 - val_accuracy: 0.5199 - val_loss: 2.3595 - val_top5: 0.8141
Epoch 3/15
2133/2133 ━━━